# Milestone 4: k-NN Artist Classification
Kevin — CSC 475, Music Maven (Group 4)

Objective 2, PI.1–PI.2:
- PI.1: Implement basic k-NN classifier that returns top-k artists given a song's feature vector
- PI.2: Calculate weighted Euclidean distance between song features and artist profiles

In [ ]:
import numpy as np
import pandas as pd

## PI.2: Weighted Euclidean Distance (Song vs Artist Profiles)
The classifier needs to measure how close a song's features are to each artist profile.
We use weighted Euclidean distance:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{N} w_i (x_i - y_i)^2}$$

Features must be normalized first so tempo (~60-200) doesn't dominate energy (0-1).

In [ ]:
# features we care about
MIR_FEATURES = ["tempo", "energy", "valence", "danceability", "key", "mode"]

# scale features to [0,1] (minmax) or mean=0/std=1 (zscore), returns (normalized_matrix, params_dict) so we can reuse on queries later
def normalize_features(X, method="minmax"):
    X = np.array(X, dtype=float)

    if method == "minmax":
        mins = X.min(axis=0)
        maxs = X.max(axis=0)
        ranges = maxs - mins
        with np.errstate(invalid="ignore", divide="ignore"):
            X_norm = np.where(ranges == 0, 0.0, (X - mins) / ranges)
        params = {"mins": mins, "maxs": maxs}
        
    elif method == "zscore":
        means = X.mean(axis=0)
        stds = X.std(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            X_norm = np.where(stds == 0, 0.0, (X - means) / stds)
        params = {"means": means, "stds": stds}
        
    else:
        raise ValueError(f"Unknown method '{method}'.")

    return X_norm, params

# apply previously fitted normalization params to a new vector
def apply_normalization(x, params, method="minmax"):
    x = np.array(x, dtype=float)

    if method == "minmax":
        ranges = params["maxs"] - params["mins"]
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(ranges == 0, 0.0, (x - params["mins"]) / ranges)

    elif method == "zscore":
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(params["stds"] == 0, 0.0, (x - params["means"]) / params["stds"])

    else:
        raise ValueError(f"Unknown method '{method}'. Use 'minmax' or 'zscore'.")

# normalize weights to sum to 1, or return uniform if None
def prepare_weights(weights, n_features):
    if weights is None:
        return np.full(n_features, 1.0 / n_features)
    w = np.array(weights, dtype=float)
    total = w.sum()
    if total == 0:
        return np.full(n_features, 1.0 / n_features)
    return w / total

# weighted euclidean distance from query to all candidates (vectorized)
def weighted_euclidean_batch(query, candidates, weights=None):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)
    w = prepare_weights(weights, query.shape[0])
    diff = candidates - query
    return np.sqrt((w * diff * diff).sum(axis=1))

## PI.1: k-NN Artist Classifier
Given a song's feature vector, find its k nearest neighbours in the training set and perform a majority vote to predict the artist

In [ ]:
class ArtistKNNClassifier:
    # k-NN classifier that maps a song's features to the most likely artist(s)

    def __init__(self, profiles_df, k=5, features=None, weights=None, normalize="minmax"):
        self.k = k
        self.features = features if features is not None else list(MIR_FEATURES)
        self.normalize_method = normalize
        self._profiles_df = profiles_df.reset_index(drop=True)

        # build weight vector from dict
        self._weight_vec = None
        if weights is not None:
            self._weight_vec = np.array([weights.get(f, 0.0) for f in self.features], dtype=float)
            train_df = self._profiles_df
            self._train_df = train_df
            raw_matrix = train_df[self.features].to_numpy(dtype=float)
            self._artist_ids = train_df["artist_id"].to_numpy()
            self._artist_names = train_df["artist_name"].to_numpy()

        # normalize training matrix and save params for queries
        self._norm_matrix, self._norm_params = normalize_features(raw_matrix, method=normalize)

    # classify a song: returns list of dicts with artist_id, votes, probability, avg_distance
    def classify(self, song_features, k=None):
        
        k = k if k is not None else self.k

        # normalize the query
        if isinstance(song_features, np.ndarray):
            raw_vec = song_features.astype(float)
        elif isinstance(song_features, (dict, pd.Series)):
            raw_vec = np.array([float(song_features[f]) for f in self.features], dtype=float)
        else:
            raw_vec = np.array(song_features, dtype=float)

        query_vec = apply_normalization(raw_vec, self._norm_params, method=self.normalize_method)

        # compute distance
        distances = weighted_euclidean_batch(query_vec, self._norm_matrix, self._weight_vec)

        # majority vote on k nearest neighbours
        return self._majority_vote(distances, k)
    
    # aggregate k nearest neighbours into artist predictions
    def _majority_vote(self, distances, k):
        # exclude any inf-masked entries
        finite_mask = np.isfinite(distances)
        finite_indices = np.where(finite_mask)[0]
        if len(finite_indices) == 0:
            return []

        effective_k = min(k, len(finite_indices))
        local_order = np.argsort(distances[finite_indices])[:effective_k]
        nn_indices = finite_indices[local_order]
        nn_distances = distances[nn_indices]
        nn_artist_ids = self._artist_ids[nn_indices]
        nn_artist_names = self._artist_names[nn_indices]

        # count votes and average distance per artist
        artist_data = {}
        for aid, aname, dist in zip(nn_artist_ids, nn_artist_names, nn_distances):
            if aid not in artist_data:
                artist_data[aid] = {"artist_id": aid, "artist_name": aname, "votes": 0, "total_dist": 0.0}
            artist_data[aid]["votes"] += 1
            artist_data[aid]["total_dist"] += dist

        results = []
        for entry in artist_data.values():
            v = entry["votes"]
            results.append({
                "artist_id": entry["artist_id"],
                "artist_name": entry["artist_name"],
                "votes": v,
                "probability": v / effective_k,
                "avg_distance": entry["total_dist"] / v,
            })

        # sort: most votes first, then closest distance
        results.sort(key=lambda x: (-x["votes"], x["avg_distance"]))
        return results

In [ ]:
# artist-profile demo, to be done once profiles are ready
# clf_profile = ArtistKNNClassifier(profiles, k=5)

# pick a test song
# test_song = songs.iloc[100]
# print(f"Test song: {test_song['song_id']} by {test_song['artist_name']} ({test_song['artist_id']})")
# print(f"Genre: {test_song['genre']}\n")

# predictions = clf_profile.classify(test_song)
# print("Artist-profile mode (k=5):")
# print(pd.DataFrame(predictions))